# E022 — Audio MFCC+LPCC Score-level Fusion

MFCC (E008) and LPCC (E020) show complementary per-fold errors:
- Fold 0: MFCC wins (3.47% vs 9.17%)
- Fold 1: LPCC wins (0.83% vs 8.33%)
- Fold 2: Both strong (0.83% vs 0.00%)

This notebook:
1. Runs LOSO CV collecting OOF scores for **both** MFCC and LPCC systems simultaneously
2. Calibrates each with Platt (LogReg C=1e6, class_weight='balanced')
3. Grid-searches optimal weight w ∈ [0,1] for `score = w·mfcc_cal + (1-w)·lpcc_cal`
4. Reports per-fold EER and overall metrics vs E008 / E020 references

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
    "mfcc":      "#E67E22",
    "lpcc":      "#8E44AD",
    "fusion":    "#E74C3C",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67

# Reference numbers from E008 and E020
E008_REF = {"fold_eers": [3.47, 8.33, 0.83], "mean": 4.21, "std": 3.11, "min_dcf_mean": 0.0509}
E020_REF = {"fold_eers": [9.17, 0.83, 0.00], "mean": 3.33, "std": 4.14, "min_dcf_mean": 0.0333}

print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")
print(f"SEED={SEED}")

## 1. Feature extraction functions

Both MFCC and LPCC yield 39 features/frame (13 static + Δ + ΔΔ + CMN).
MFCC: mel-warped DCT representation (E008).
LPCC: all-pole vocal tract cepstrum via FFT (E020).

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def extract_mfcc(y: np.ndarray, sr: int, n_mfcc: int = 13) -> np.ndarray:
    """Extract MFCC+Δ+ΔΔ+CMN. Returns (T, 39)."""
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.vstack([mfcc, delta, delta2]).T   # (T, 39)
    feat  -= feat.mean(axis=0)                     # CMN
    return feat


def extract_lpcc(y: np.ndarray, sr: int, order: int = 12, n_cep: int = 13,
                 hop_length: int = 160, win_length: int = 400) -> np.ndarray:
    """Extract LPCC+Δ+ΔΔ+CMN. Returns (T, 39)."""
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    lpcc_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        lpcc_frames.append(cep)
    lpcc = np.array(lpcc_frames, dtype=np.float32)
    delta  = librosa.feature.delta(lpcc.T).T
    delta2 = librosa.feature.delta(lpcc.T, order=2).T
    feat   = np.hstack([lpcc, delta, delta2])    # (T, 39)
    feat  -= feat.mean(axis=0)                    # CMN
    return feat


def aug_noise(y: np.ndarray, rng: np.random.Generator, snr_db: float = 20.0) -> np.ndarray:
    p = np.mean(y**2) + 1e-10
    return y + rng.normal(0, np.sqrt(p / 10**(snr_db/10)), len(y)).astype(y.dtype)


def aug_speed(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return librosa.effects.time_stretch(y, rate=rng.uniform(0.9, 1.1))


def load_augmented(wav_path: Path, rng: np.random.Generator):
    """Load WAV, return list of (y, sr) — original + noisy + speed-perturbed."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    return [
        (y, sr),
        (aug_noise(y, rng=rng, snr_db=20.0), sr),
        (aug_speed(y, rng=rng), sr),
    ]


# Sanity check
sample_row = manifest[manifest.label == 1].iloc[0]
y_tmp, sr_tmp = librosa.load(find_wav(sample_row["stem"], DATA), sr=None, mono=True)
f_mfcc = extract_mfcc(y_tmp, sr_tmp)
f_lpcc = extract_lpcc(y_tmp, sr_tmp)
print(f"MFCC shape: {f_mfcc.shape}  (T x 39)")
print(f"LPCC shape: {f_lpcc.shape}  (T x 39)")
print("Feature extraction functions OK.")

## 2. UBM + MAP functions

In [ ]:
def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    log_prob  = ubm._estimate_log_prob(X_target)
    log_resp  = log_prob + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)
    mu_hat    = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha     = n_k / (n_k + r)
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


def score_utterance(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture,
                    feat_fn) -> float:
    """LLR score for one utterance using the provided feature extractor."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat  = feat_fn(y, sr)
    return float((adapted.score_samples(feat) - ubm.score_samples(feat)).mean())


print("UBM+MAP functions defined.")

## 3. LOSO CV — collecting OOF scores for both systems

Each fold:
1. Extract augmented MFCC frames from train → train UBM_mfcc + MAP_mfcc
2. Extract augmented LPCC frames from train → train UBM_lpcc + MAP_lpcc
3. Score val fold (original WAVs only) with each system → `oof_mfcc`, `oof_lpcc`

Both UBMs are trained **within** the fold — no leakage across folds.

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0

oof_mfcc = np.full(len(manifest), np.nan)
oof_lpcc = np.full(len(manifest), np.nan)
fold_results = []  # per-fold EERs for both systems

print("Running LOSO CV — MFCC and LPCC systems in parallel per fold")
print("=" * 65)

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    rng = np.random.default_rng(SEED + fold_id)

    print(f"\nFold {fold_id}: {len(train_df)} train / {len(val_df)} val")

    # --- Collect augmented frames from train fold ---
    # Track labels separately for each feature type — frame counts per utterance differ
    mfcc_frames, mfcc_labels = [], []
    lpcc_frames, lpcc_labels = [], []

    for _, row in train_df.iterrows():
        wav_path = find_wav(row["stem"], DATA)
        for y_aug, sr in load_augmented(wav_path, rng):
            fm = extract_mfcc(y_aug, sr)
            fl = extract_lpcc(y_aug, sr)
            mfcc_frames.append(fm)
            lpcc_frames.append(fl)
            mfcc_labels.extend([row["label"]] * len(fm))
            lpcc_labels.extend([row["label"]] * len(fl))

    X_mfcc = np.vstack(mfcc_frames)
    X_lpcc = np.vstack(lpcc_frames)
    y_mfcc = np.array(mfcc_labels)
    y_lpcc = np.array(lpcc_labels)

    X_mfcc_nt = X_mfcc[y_mfcc == 0]
    X_mfcc_t  = X_mfcc[y_mfcc == 1]
    X_lpcc_nt = X_lpcc[y_lpcc == 0]
    X_lpcc_t  = X_lpcc[y_lpcc == 1]

    print(f"  MFCC: {len(X_mfcc_t)} target frames, {len(X_mfcc_nt)} non-target frames")
    print(f"  LPCC: {len(X_lpcc_t)} target frames, {len(X_lpcc_nt)} non-target frames")

    # --- Train UBM + MAP for MFCC ---
    ubm_mfcc     = train_ubm(X_mfcc_nt, n_components=UBM_COMPONENTS, seed=SEED)
    adapted_mfcc = map_adapt(ubm_mfcc, X_mfcc_t, r=MAP_R)
    print(f"  MFCC UBM+MAP trained")

    # --- Train UBM + MAP for LPCC ---
    ubm_lpcc     = train_ubm(X_lpcc_nt, n_components=UBM_COMPONENTS, seed=SEED)
    adapted_lpcc = map_adapt(ubm_lpcc, X_lpcc_t, r=MAP_R)
    print(f"  LPCC UBM+MAP trained")

    # --- Score val fold (original WAVs only) ---
    for idx, row in val_df.iterrows():
        wav_path = find_wav(row["stem"], DATA)
        oof_mfcc[idx] = score_utterance(wav_path, adapted_mfcc, ubm_mfcc, extract_mfcc)
        oof_lpcc[idx] = score_utterance(wav_path, adapted_lpcc, ubm_lpcc, extract_lpcc)

    # Per-fold EER for each system
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    s_mfcc = oof_mfcc[val_idx]
    s_lpcc = oof_lpcc[val_idx]

    eer_m, _ = compute_eer(s_mfcc[val_labels==1], s_mfcc[val_labels==0])
    eer_l, _ = compute_eer(s_lpcc[val_labels==1], s_lpcc[val_labels==0])
    dcf_m, _ = compute_min_dcf(s_mfcc[val_labels==1], s_mfcc[val_labels==0])
    dcf_l, _ = compute_min_dcf(s_lpcc[val_labels==1], s_lpcc[val_labels==0])

    fold_results.append({
        "fold": fold_id,
        "eer_mfcc": eer_m, "dcf_mfcc": dcf_m,
        "eer_lpcc": eer_l, "dcf_lpcc": dcf_l,
    })

    print(f"  Fold {fold_id}: MFCC EER={eer_m*100:.2f}%  LPCC EER={eer_l*100:.2f}%")

print("\n" + "=" * 65)
print("CV complete.")
print(f"OOF coverage: MFCC {(~np.isnan(oof_mfcc)).sum()}/{len(manifest)}, "
      f"LPCC {(~np.isnan(oof_lpcc)).sum()}/{len(manifest)}")

## 4. Platt calibration

Fit LogisticRegression (C=1e6, class_weight='balanced') on all OOF scores
for each system separately. This scales scores to log-probability space
and accounts for prior imbalance.

In [ ]:
def platt_calibrate(scores: np.ndarray, labels: np.ndarray) -> tuple:
    """Fit Platt calibrator. Returns (calibrator, calibrated_scores)."""
    cal = LogisticRegression(C=1e6, max_iter=2000, class_weight="balanced", random_state=SEED)
    cal.fit(scores.reshape(-1, 1), labels)
    cal_scores = cal.decision_function(scores.reshape(-1, 1))
    return cal, cal_scores


# Raw OOF EERs before calibration
eer_mfcc_raw, thr_mfcc_raw = compute_eer(oof_mfcc[y_all==1], oof_mfcc[y_all==0])
eer_lpcc_raw, thr_lpcc_raw = compute_eer(oof_lpcc[y_all==1], oof_lpcc[y_all==0])
print(f"Pre-calibration OOF EER:")
print(f"  MFCC: {eer_mfcc_raw*100:.2f}%  threshold={thr_mfcc_raw:.4f}")
print(f"  LPCC: {eer_lpcc_raw*100:.2f}%  threshold={thr_lpcc_raw:.4f}")

# Calibrate each system separately
cal_mfcc, oof_mfcc_cal = platt_calibrate(oof_mfcc, y_all)
cal_lpcc, oof_lpcc_cal = platt_calibrate(oof_lpcc, y_all)

eer_mfcc_cal, thr_mfcc_cal = compute_eer(oof_mfcc_cal[y_all==1], oof_mfcc_cal[y_all==0])
eer_lpcc_cal, thr_lpcc_cal = compute_eer(oof_lpcc_cal[y_all==1], oof_lpcc_cal[y_all==0])
print(f"\nPost-calibration OOF EER (should be same EER, threshold shifted to ~0):")
print(f"  MFCC cal: {eer_mfcc_cal*100:.2f}%  threshold={thr_mfcc_cal:.4f}")
print(f"  LPCC cal: {eer_lpcc_cal*100:.2f}%  threshold={thr_lpcc_cal:.4f}")

print(f"\nPlatt params:")
print(f"  MFCC: slope={cal_mfcc.coef_[0,0]:.4f}, intercept={cal_mfcc.intercept_[0]:.4f}")
print(f"  LPCC: slope={cal_lpcc.coef_[0,0]:.4f}, intercept={cal_lpcc.intercept_[0]:.4f}")

In [ ]:
# Visualize score distributions before and after calibration
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for row_i, (raw_scores, cal_scores, name, color) in enumerate([
    (oof_mfcc, oof_mfcc_cal, "MFCC (E008)", COLORS["mfcc"]),
    (oof_lpcc, oof_lpcc_cal, "LPCC (E020)", COLORS["lpcc"]),
]):
    for col_i, (scores, label_suffix, thr) in enumerate([
        (raw_scores, "raw", thr_mfcc_raw if row_i==0 else thr_lpcc_raw),
        (cal_scores, "Platt calibrated", thr_mfcc_cal if row_i==0 else thr_lpcc_cal),
    ]):
        ax = axes[row_i, col_i]
        eer_v, eer_t = compute_eer(scores[y_all==1], scores[y_all==0])
        bins = np.linspace(np.percentile(scores, 1), np.percentile(scores, 99), 40)
        ax.hist(scores[y_all==0], bins=bins, alpha=0.6, color=COLORS["nontarget"],
                label="non-target", density=True)
        ax.hist(scores[y_all==1], bins=bins, alpha=0.75, color=COLORS["target"],
                label="target", density=True)
        ax.axvline(eer_t, color=COLORS["green"], ls="--", lw=2,
                   label=f"EER thr={eer_t:.3f}")
        ax.axvline(0, color="black", ls=":", lw=1.5, alpha=0.7, label="0")
        ax.set_title(f"{name} — {label_suffix}\nEER={eer_v*100:.2f}%")
        ax.set_xlabel("Score")
        ax.legend(fontsize=8)

plt.suptitle("Calibration: score distributions before and after Platt", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Grid search: optimal MFCC weight w

score = w · mfcc_cal + (1−w) · lpcc_cal

w=1 → MFCC only, w=0 → LPCC only. Grid over 101 values, minimize OOF EER.

In [ ]:
weights = np.linspace(0, 1, 101)
grid_eers = []
grid_dcfs = []

for w in weights:
    fused = w * oof_mfcc_cal + (1 - w) * oof_lpcc_cal
    eer_f, _ = compute_eer(fused[y_all==1], fused[y_all==0])
    dcf_f, _ = compute_min_dcf(fused[y_all==1], fused[y_all==0])
    grid_eers.append(eer_f)
    grid_dcfs.append(dcf_f)

best_w_idx = int(np.argmin(grid_eers))
best_w     = weights[best_w_idx]
best_eer   = grid_eers[best_w_idx]
best_dcf   = grid_dcfs[best_w_idx]

oof_fused = best_w * oof_mfcc_cal + (1 - best_w) * oof_lpcc_cal
eer_fused_oof, thr_fused = compute_eer(oof_fused[y_all==1], oof_fused[y_all==0])
dcf_fused_oof, _         = compute_min_dcf(oof_fused[y_all==1], oof_fused[y_all==0])

print(f"Grid search results:")
print(f"  Best w = {best_w:.2f}  (MFCC weight={best_w:.2f}, LPCC weight={1-best_w:.2f})")
print(f"  Fusion OOF EER = {best_eer*100:.2f}%")
print(f"  Fusion OOF min-DCF = {best_dcf:.4f}")
print(f"  Fusion threshold = {thr_fused:.4f}")
print(f"\nFor reference:")
print(f"  MFCC alone (w=1.0): OOF EER = {eer_mfcc_cal*100:.2f}%")
print(f"  LPCC alone (w=0.0): OOF EER = {eer_lpcc_cal*100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(weights, [e*100 for e in grid_eers], color=COLORS["fusion"], lw=2.5, label="Fused EER")

# Vertical lines for reference points
ax.axvline(best_w, color=COLORS["green"], ls="--", lw=2.5,
           label=f"best w={best_w:.2f}  EER={best_eer*100:.2f}%")
ax.axvline(0.0, color=COLORS["lpcc"], ls=":", lw=2,
           label=f"w=0 (LPCC only)  EER={eer_lpcc_cal*100:.2f}%")
ax.axvline(1.0, color=COLORS["mfcc"], ls=":", lw=2,
           label=f"w=1 (MFCC only)  EER={eer_mfcc_cal*100:.2f}%")

# Horizontal reference lines
ax.axhline(eer_mfcc_cal*100, color=COLORS["mfcc"], ls="-.", lw=1, alpha=0.5)
ax.axhline(eer_lpcc_cal*100, color=COLORS["lpcc"], ls="-.", lw=1, alpha=0.5)
ax.axhline(best_eer*100, color=COLORS["green"], ls="-.", lw=1, alpha=0.5)

ax.set_xlabel("MFCC weight w  (LPCC weight = 1−w)")
ax.set_ylabel("OOF EER [%]")
ax.set_title("E022 — Grid search: optimal MFCC/LPCC weight (OOF EER)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Results table

Per-fold EER for both subsystems, plus OOF fusion result.

In [ ]:
# Per-fold EERs from CV
eers_mfcc_cv = [r["eer_mfcc"]*100 for r in fold_results]
eers_lpcc_cv = [r["eer_lpcc"]*100 for r in fold_results]
dcfs_mfcc_cv = [r["dcf_mfcc"] for r in fold_results]
dcfs_lpcc_cv = [r["dcf_lpcc"] for r in fold_results]

# Per-fold EER for fusion: re-compute using per-fold calibrated val scores
# Note: OOF calibration is global — per-fold breakdown of fused OOF scores
eers_fused_cv = []
dcfs_fused_cv = []
for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    fused_val = oof_fused[val_idx]
    eer_f, _ = compute_eer(fused_val[val_labels==1], fused_val[val_labels==0])
    dcf_f, _ = compute_min_dcf(fused_val[val_labels==1], fused_val[val_labels==0])
    eers_fused_cv.append(eer_f*100)
    dcfs_fused_cv.append(dcf_f)

mean_mfcc = np.mean(eers_mfcc_cv)
std_mfcc  = np.std(eers_mfcc_cv)
mean_lpcc = np.mean(eers_lpcc_cv)
std_lpcc  = np.std(eers_lpcc_cv)
mean_fus  = np.mean(eers_fused_cv)
std_fus   = np.std(eers_fused_cv)
mean_dcf_mfcc = np.mean(dcfs_mfcc_cv)
mean_dcf_lpcc = np.mean(dcfs_lpcc_cv)
mean_dcf_fus  = np.mean(dcfs_fused_cv)

print("Results table (per-fold EER from CV):")
print()
hdr = f"{'System':<22} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}"
print(hdr)
print("-" * len(hdr))
print(f"{'MFCC alone (E008)':<22} {eers_mfcc_cv[0]:>8.2f} {eers_mfcc_cv[1]:>8.2f} "
      f"{eers_mfcc_cv[2]:>8.2f} {mean_mfcc:>8.2f} {std_mfcc:>8.2f} {mean_dcf_mfcc:>9.4f}")
print(f"{'LPCC alone (E020)':<22} {eers_lpcc_cv[0]:>8.2f} {eers_lpcc_cv[1]:>8.2f} "
      f"{eers_lpcc_cv[2]:>8.2f} {mean_lpcc:>8.2f} {std_lpcc:>8.2f} {mean_dcf_lpcc:>9.4f}")
print(f"{'MFCC+LPCC fusion':<22} {eers_fused_cv[0]:>8.2f} {eers_fused_cv[1]:>8.2f} "
      f"{eers_fused_cv[2]:>8.2f} {mean_fus:>8.2f} {std_fus:>8.2f} {mean_dcf_fus:>9.4f}")
print("-" * len(hdr))
print()
print(f"Optimal w = {best_w:.2f}  (MFCC weight, LPCC weight = {1-best_w:.2f})")
print(f"OOF overall — Fusion EER={eer_fused_oof*100:.2f}%  min-DCF={dcf_fused_oof:.4f}  thr={thr_fused:.4f}")
print()
delta_vs_mfcc = mean_fus - mean_mfcc
delta_vs_lpcc = mean_fus - mean_lpcc
print(f"Fusion vs MFCC: {delta_vs_mfcc:+.2f}pp")
print(f"Fusion vs LPCC: {delta_vs_lpcc:+.2f}pp")

## 7. Score distributions (2×2)

Per-fold fused score distributions + OOF overall.

In [ ]:
# Collect per-fold fused scores and labels
fold_fused_data = []
for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    fold_fused_data.append({
        "scores": oof_fused[val_idx],
        "labels": val_labels,
    })

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

all_finite = oof_fused[~np.isnan(oof_fused)]
bin_edges = np.linspace(np.percentile(all_finite, 1), np.percentile(all_finite, 99), 35)

for i, (ax, fdata) in enumerate(zip(axes[:3], fold_fused_data)):
    s, l = fdata["scores"], fdata["labels"]
    eer_f, thr_f = compute_eer(s[l==1], s[l==0])
    ax.hist(s[l==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
    ax.hist(s[l==1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
    ax.axvline(thr_f, color=COLORS["green"], ls="--", lw=2, label=f"thresh={thr_f:.3f}")
    ax.axvline(0, color="black", ls=":", lw=1.5, alpha=0.5, label="0")
    ax.set_title(f"Fold {i}  —  Fused EER={eer_f*100:.1f}%")
    ax.set_xlabel("Fused score")
    ax.legend(fontsize=8)

ax = axes[3]
ax.hist(oof_fused[y_all==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
ax.hist(oof_fused[y_all==1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
ax.axvline(thr_fused, color=COLORS["green"], ls="--", lw=2, label=f"OOF thresh={thr_fused:.3f}")
ax.axvline(0, color="black", ls=":", lw=1.5, alpha=0.5, label="0")
ax.set_title(f"OOF overall  —  EER={eer_fused_oof*100:.1f}%")
ax.set_xlabel("Fused score")
ax.legend(fontsize=8)

plt.suptitle(f"E022 — MFCC+LPCC fusion score distributions (w={best_w:.2f})", fontsize=12)
plt.tight_layout()
plt.show()

## 8. DET curves — MFCC alone, LPCC alone, fusion

In [ ]:
ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos    = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]

systems_det = [
    (oof_mfcc_cal, "MFCC alone (E008 cal)", COLORS["mfcc"], "--", 1.5),
    (oof_lpcc_cal, "LPCC alone (E020 cal)", COLORS["lpcc"], "--", 1.5),
    (oof_fused,    f"MFCC+LPCC fusion w={best_w:.2f}", COLORS["fusion"], "-", 2.5),
]

fig, ax = plt.subplots(figsize=(7, 7))

for scores, name, color, ls, lw in systems_det:
    valid = ~np.isnan(scores)
    eer_d, _ = compute_eer(scores[valid & (y_all==1)], scores[valid & (y_all==0)])
    fpr, tpr, _ = roc_curve(y_all[valid], scores[valid])
    far_c = np.clip(fpr, 1e-4, 1-1e-4)
    frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
            color=color, lw=lw, ls=ls,
            label=f"{name}  EER={eer_d*100:.1f}%")

ax.plot(tick_pos, tick_pos, "k--", lw=1, label="EER line")
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET curves — MFCC, LPCC, and fusion (OOF)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 9. Final summary

In [ ]:
print("=" * 65)
print("E022 — MFCC+LPCC audio fusion — FINAL RESULTS")
print("=" * 65)
print()
print(f"{'System':<28} {'Mean EER±std':>16} {'min-DCF':>9}")
print("-" * 56)
print(f"{'MFCC alone (E008 ref)':<28} {mean_mfcc:>6.2f} ± {std_mfcc:<6.2f}   {mean_dcf_mfcc:>8.4f}")
print(f"{'LPCC alone (E020 ref)':<28} {mean_lpcc:>6.2f} ± {std_lpcc:<6.2f}   {mean_dcf_lpcc:>8.4f}")
print(f"{'MFCC+LPCC fusion':<28} {mean_fus:>6.2f} ± {std_fus:<6.2f}   {mean_dcf_fus:>8.4f}")
print("-" * 56)
print()
print(f"Optimal MFCC weight w = {best_w:.2f}  (LPCC weight = {1-best_w:.2f})")
print(f"OOF overall EER        = {eer_fused_oof*100:.2f}%")
print(f"OOF overall min-DCF    = {dcf_fused_oof:.4f}")
print(f"OOF threshold          = {thr_fused:.4f}")
print()
better_than_mfcc = mean_fus < mean_mfcc
better_than_lpcc = mean_fus < mean_lpcc
print(f"Fusion beats MFCC: {better_than_mfcc}  ({delta_vs_mfcc:+.2f}pp)")
print(f"Fusion beats LPCC: {better_than_lpcc}  ({delta_vs_lpcc:+.2f}pp)")
print(f"Std reduction vs MFCC: {std_mfcc - std_fus:+.2f}pp  ({'reduced' if std_fus < std_mfcc else 'increased'})")
print(f"Std reduction vs LPCC: {std_lpcc - std_fus:+.2f}pp  ({'reduced' if std_fus < std_lpcc else 'increased'})")